In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)


# ============================================
# PROJECT ROOT
# ============================================

ROOT = Path(
    r"D:\My_Project\Analyse\capston Project_2-Global Supply Chain Risk & Logistics"
)


# ============================================
# INPUT DATA
# ============================================

DATA_PATH = (
    ROOT
    / "data"
    / "interim"
    / "feature_engineered_data.csv"
)


# ============================================
# PROCESSED DATA PATH
# ============================================

PROCESSED_PATH = (
    ROOT
    / "data"
    / "processed"
)


# ============================================
# MODEL PATH
# ============================================

MODEL_PATH = (
    ROOT
    / "models"
)


# ============================================
# CREATE FOLDERS
# ============================================

PROCESSED_PATH.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_PATH.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================
# LOAD FEATURE-ENGINEERED DATA
# ============================================

df = pd.read_csv(DATA_PATH)


# ============================================
# CHECK DATA
# ============================================

print("Data path:")
print(DATA_PATH)

print("\nShape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

display(df.head())

Data path:
D:\My_Project\Analyse\capston Project_2-Global Supply Chain Risk & Logistics\data\interim\feature_engineered_data.csv

Shape:
(5000, 13)

Columns:
['date', 'origin_port', 'destination_port', 'transport_mode', 'product_category', 'distance_km', 'weight_mt', 'fuel_price_index', 'geopolitical_risk_score', 'weather_condition', 'carrier_reliability_score', 'lead_time_days', 'disruption']


,date,origin_port,destination_port,transport_mode,product_category,distance_km,weight_mt,fuel_price_index,geopolitical_risk_score,weather_condition,carrier_reliability_score,lead_time_days,disruption
0,2025-10-16,Singapore,Los Angeles,Rail,Textiles,5930.83,197.42,2.43,5.0,Hurricane,0.865,41.39,1
1,2024-04-24,Singapore,Shanghai,Rail,Automotive,14285.36,237.24,2.30,7.5,Storm,0.592,40.92,1
2,2024-01-26,Rotterdam,Los Angeles,Rail,Perishables,11113.91,427.42,1.78,5.6,Rain,0.673,11.54,0
3,2024-10-08,Busan,Hamburg,Rail,Electronics,9180.55,170.66,3.20,0.8,Hurricane,0.832,53.13,1
4,2024-09-07,Busan,Singapore,Air,Perishables,2762.27,434.96,2.77,1.9,Fog,0.741,0.50,1


In [2]:
if "disruption" not in df.columns:

    raise ValueError(
        "Target column 'disruption' not found."
    )

df = df.dropna(
    subset=["disruption"]
)

y = df["disruption"].astype(int)

X = df.drop(
    columns=["disruption"]
)

In [3]:
all_missing_columns = [
    column
    for column in X.columns
    if X[column].isna().all()
]

X = X.drop(
    columns=all_missing_columns
)

print(
    "Removed all-missing columns:",
    all_missing_columns
)

Removed all-missing columns: []


In [4]:
high_cardinality_columns = []

for column in X.select_dtypes(
    include="object"
).columns:

    ratio = (
        X[column].nunique(
            dropna=False
        )
        /
        len(X)
    )

    if ratio > 0.95:

        high_cardinality_columns.append(
            column
        )

X = X.drop(
    columns=high_cardinality_columns
)

print(
    "Removed high-cardinality columns:",
    high_cardinality_columns
)

Removed high-cardinality columns: []


C:\Users\USER\AppData\Local\Temp\ipykernel_23396\3915992002.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for column in X.select_dtypes(


In [5]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.40,
    random_state=42,
    stratify=y
)

X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Train:", X_train.shape)
print("Validation:", X_validation.shape)
print("Test:", X_test.shape)

Train: (3000, 12)
Validation: (1000, 12)
Test: (1000, 12)


In [6]:
train_df = X_train.copy()
train_df["disruption"] = y_train.values

validation_df = X_validation.copy()
validation_df["disruption"] = y_validation.values

test_df = X_test.copy()
test_df["disruption"] = y_test.values

train_df.to_csv(
    PROCESSED_PATH / "D:\\My_Project\\Analyse\\capston Project_2-Global Supply Chain Risk & Logistics\\data\\processed\\train.csv",
    index=False
)

validation_df.to_csv(
    PROCESSED_PATH / "D:\\My_Project\\Analyse\\capston Project_2-Global Supply Chain Risk & Logistics\\data\\processed\\validation.csv",
    index=False
)

test_df.to_csv(
    PROCESSED_PATH / "D:\\My_Project\\Analyse\\capston Project_2-Global Supply Chain Risk & Logistics\\data\\processed\\test.csv",
    index=False
)

print("Datasets saved.")

Datasets saved.


In [7]:
numeric_features = (
    X_train
    .select_dtypes(include=np.number)
    .columns
    .tolist()
)

categorical_features = (
    X_train
    .select_dtypes(exclude=np.number)
    .columns
    .tolist()
)

print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numeric features:
['distance_km', 'weight_mt', 'fuel_price_index', 'geopolitical_risk_score', 'carrier_reliability_score', 'lead_time_days']

Categorical features:
['date', 'origin_port', 'destination_port', 'transport_mode', 'product_category', 'weather_condition']


In [8]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

In [12]:
import joblib

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ],
    remainder="drop"
)

# Fit ONLY on training data
preprocessor.fit(X_train)

# Save preprocessor
joblib.dump(
    preprocessor,
    MODEL_PATH / "preprocessor.pkl"
)

print("Preprocessor saved:")
print(
    (MODEL_PATH / "preprocessor.pkl").resolve()
)

Preprocessor saved:
D:\My_Project\Analyse\capston Project_2-Global Supply Chain Risk & Logistics\models\preprocessor.pkl
